## Sentiment Analysis Using RNN

### Import required packages

In [13]:
import numpy as np
import torch

### get the data

In [14]:
reviews = [
    "I love this product it's amazing!",
    "Excellent service highly recommended!",
    "The best experience ever!",
    "Great quality and value for money.",
    "Absolutely fantastic would buy again!",
    "Superb performance definitely worth it.",
    "Loved every moment of using this!",
    "Outstanding customer support, thank you!",
    "A must-have item highly satisfied!",
    "Perfectly meets all my expectations!",

    "This product is terrible don't buy it!",
    "Disappointing experience not worth the money.",
    "The worst service I've ever encountered.",
    "Poor quality completely broken.",
    "Not worth the time or money.",
    "Horrible customer support very frustrating.",
    "I regret purchasing this item.",
    "Badly designed doesn't work as intended.",
    "Extremely dissatisfied with the product.",
    "Complete waste of money avoid at all costs."
]

# class labels
labels = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

### text pre-processing

In [15]:
# this is an assignment to implement the text pre-processing

### build vocabulary

In [16]:
# set the max length of every word in vocabulary
MAX_LENGTH = 10

# create empty vocabulary
vocabulary = {
    # we will keep the tokens in each review with fixed length
    # if the number of tokens is smaller than the fixed length, we will add the padding at the end
    "<PAD>": 0
}

In [17]:
# split all the reviews into tokens (words)
for review in reviews:
    for word in review.split():

        # check if the word is already present in the vocabulary
        if word not in vocabulary:
            vocabulary[word] = len(vocabulary)

# print vocubulary
print(vocabulary)

{'<PAD>': 0, 'I': 1, 'love': 2, 'this': 3, 'product': 4, "it's": 5, 'amazing!': 6, 'Excellent': 7, 'service': 8, 'highly': 9, 'recommended!': 10, 'The': 11, 'best': 12, 'experience': 13, 'ever!': 14, 'Great': 15, 'quality': 16, 'and': 17, 'value': 18, 'for': 19, 'money.': 20, 'Absolutely': 21, 'fantastic': 22, 'would': 23, 'buy': 24, 'again!': 25, 'Superb': 26, 'performance': 27, 'definitely': 28, 'worth': 29, 'it.': 30, 'Loved': 31, 'every': 32, 'moment': 33, 'of': 34, 'using': 35, 'this!': 36, 'Outstanding': 37, 'customer': 38, 'support,': 39, 'thank': 40, 'you!': 41, 'A': 42, 'must-have': 43, 'item': 44, 'satisfied!': 45, 'Perfectly': 46, 'meets': 47, 'all': 48, 'my': 49, 'expectations!': 50, 'This': 51, 'is': 52, 'terrible': 53, "don't": 54, 'it!': 55, 'Disappointing': 56, 'not': 57, 'the': 58, 'worst': 59, "I've": 60, 'ever': 61, 'encountered.': 62, 'Poor': 63, 'completely': 64, 'broken.': 65, 'Not': 66, 'time': 67, 'or': 68, 'Horrible': 69, 'support': 70, 'very': 71, 'frustrating

### encode all the reviews

In [18]:
# ecode the view in numeric format
def encode(review):

    # split the review in tokens (words)
    tokens = [vocabulary[word] for word in review.split()]

    # add padding if required
    while len(tokens) < MAX_LENGTH:
        tokens.append(0)

    return tokens

In [19]:
# encode all the reviews
encoded_reviews = [encode(review) for review in reviews]
encoded_reviews

[[1, 2, 3, 4, 5, 6, 0, 0, 0, 0],
 [7, 8, 9, 10, 0, 0, 0, 0, 0, 0],
 [11, 12, 13, 14, 0, 0, 0, 0, 0, 0],
 [15, 16, 17, 18, 19, 20, 0, 0, 0, 0],
 [21, 22, 23, 24, 25, 0, 0, 0, 0, 0],
 [26, 27, 28, 29, 30, 0, 0, 0, 0, 0],
 [31, 32, 33, 34, 35, 36, 0, 0, 0, 0],
 [37, 38, 39, 40, 41, 0, 0, 0, 0, 0],
 [42, 43, 44, 9, 45, 0, 0, 0, 0, 0],
 [46, 47, 48, 49, 50, 0, 0, 0, 0, 0],
 [51, 4, 52, 53, 54, 24, 55, 0, 0, 0],
 [56, 13, 57, 29, 58, 20, 0, 0, 0, 0],
 [11, 59, 8, 60, 61, 62, 0, 0, 0, 0],
 [63, 16, 64, 65, 0, 0, 0, 0, 0, 0],
 [66, 29, 58, 67, 68, 20, 0, 0, 0, 0],
 [69, 38, 70, 71, 72, 0, 0, 0, 0, 0],
 [1, 73, 74, 3, 75, 0, 0, 0, 0, 0],
 [76, 77, 78, 79, 80, 81, 0, 0, 0, 0],
 [82, 83, 84, 58, 85, 0, 0, 0, 0, 0],
 [86, 87, 34, 88, 89, 90, 48, 91, 0, 0]]

### define a data set

In [20]:
from torch.utils.data import Dataset

# create a custom data set to have the reviews in tensor format
class SentimentDataset(Dataset):

    # initialization of dataset
    def __init__(self):
        # super.__init__(self)

        # convert and store the encoded reviews in tensor
        self.x = torch.tensor(encoded_reviews, dtype=torch.long)

        # conver and store the corresponding labels in tensor
        self.y = torch.tensor(labels, dtype=torch.float32)

    # return the length of dataset
    def __len__(self):
        return len(self.x)

    # return a value at required index position
    def __getitem__(self, index):
        return self.x[index], self.y[index]

In [21]:
# create an instance of SentimentDataset
dataset = SentimentDataset()

### create data loader

In [22]:
from torch.utils.data import DataLoader

# create a data load to load the dataset
loader = DataLoader(dataset=dataset,batch_size=2,shuffle=True)

### Create a model

In [23]:
vector1 = torch.tensor([10, 20, 30, 40])
print(vector1.dtype)

vector2 = torch.tensor([10.0, 20., 30., 40.])
print(vector2.dtype)

torch.int64
torch.float32


### define the model class

In [25]:
# create a subclass of nn.Module to represent the setiment analysis model
class SentimentAnalysis(torch.nn.Module):

    def __init__(self):
        # initialize the parent
        super().__init__()

        # add the required layer
        self.embedding = torch.nn.Embedding(
            num_embeddings=len(vocabulary),
            embedding_dim=8
        )

        # add the RNN layer
        self.rnn = torch.nn.RNN(
            input_size=8,
            hidden_size=16,
            batch_first=True
        )

        # add the linear layer
        self.linear = torch.nn.Linear(16, 1)

        # apply the sigmoid activation function to get the final result as 0 or 1
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):

        # convert the input to the embedding
        x = self.embedding(x)

        # pass these embeddings to the RNN layer
        # the hidden state is in the shape: (1, batch_size, hidden_size)
        output, hidden = self.rnn(x)

        # convert the shape of hidden state: (batch_size, hidden_size)
        hidden = hidden.squeeze(0)

        # pass the reshaped hidden state to the linear layer
        x = self.linear(hidden)

        # get the binary classification result using sigmoid
        x = self.sigmoid(x)

        # return the final result to the caller
        return x

### detect the GPU

In [26]:
device = ''

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"detected device = {device}")

detected device = cpu


### define the hyper parameters

In [27]:
# create the model instance
model = SentimentAnalysis()

# move the model to device
model = model.to(device)

In [28]:
# decide the loss function
loss_function = torch.nn.BCELoss()

# device the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# decide the epochs value
epochs = 50

### training loop

In [29]:
for epoch in range(epochs):

    # collect the total loss
    total_loss = 0

    # iterate over the data loader to get the data in a required batch_size
    for x, y in loader:

        # move x to device
        x = x.to(device)

        # move y to device
        y = y.to(device).view(-1, 1)

        # pass the input to the model and get the predictions
        # forward() of model gets invoked
        predictions = model(x)

        # calculate the loss
        loss = loss_function(predictions, y)

        # zero out the gradients
        optimizer.zero_grad()

        # calculate the gradients
        loss.backward()

        # optimize the model parameters
        optimizer.step()

        # collect the loss
        total_loss += loss.item()

    # print the loss
    print(f"epoch = {epoch}, total_loss = {total_loss}")

epoch = 0, total_loss = 7.156235456466675
epoch = 1, total_loss = 7.040121853351593
epoch = 2, total_loss = 6.701355278491974
epoch = 3, total_loss = 5.194046884775162
epoch = 4, total_loss = 3.648391142487526
epoch = 5, total_loss = 3.1163256615400314
epoch = 6, total_loss = 1.5575674697756767
epoch = 7, total_loss = 1.196832437068224
epoch = 8, total_loss = 0.4229477196931839
epoch = 9, total_loss = 0.25154420733451843
epoch = 10, total_loss = 0.1652486603707075
epoch = 11, total_loss = 0.12955592013895512
epoch = 12, total_loss = 0.10520083084702492
epoch = 13, total_loss = 0.08863824838772416
epoch = 14, total_loss = 0.07753379456698895
epoch = 15, total_loss = 0.0680684489198029
epoch = 16, total_loss = 0.06092610349878669
epoch = 17, total_loss = 0.05504699470475316
epoch = 18, total_loss = 0.05024051759392023
epoch = 19, total_loss = 0.04601634689606726
epoch = 20, total_loss = 0.04237603163346648
epoch = 21, total_loss = 0.03916228818707168
epoch = 22, total_loss = 0.0363607320

### test the model by passing unseen reviews

In [30]:
def predict(review):
    # change the mode of model to eval()
    model.eval()

    # encode the review
    encoded_review = torch.tensor([encode(review)], dtype=torch.long).to(device)

    # disable learning
    with torch.no_grad():
        # predict the sentiment for the review
        prediction = model(encoded_review)
        if prediction > 0.5:
            print(f"review is positive: [{prediction}]")
        else:
            print(f"review is negative: [{prediction}]")

In [31]:
predict("regret purchasing the item.")

review is positive: [tensor([[0.9979]])]


In [32]:
predict("Great quality")

review is positive: [tensor([[0.9992]])]


In [33]:
predict("terrible product")

review is negative: [tensor([[0.0160]])]


In [34]:
predict("highly satisfied!")

review is positive: [tensor([[0.9985]])]


In [35]:
predict("Not worth the time")

review is negative: [tensor([[0.0011]])]
